<a href="https://colab.research.google.com/github/Yossef-Dawoad/clip.cpp/blob/add_colab_notebook_example/examples/python_bindings/notebooks/clipcpp_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Downloading an image from internet

In [ ]:
!wget https://i.imgur.com/8H7XCH0.jpg -O cat.jpg

In [ ]:
from IPython.display import Image
Image('cat.jpg')

## Building Clip.cpp repo from source (not required)

In [ ]:
!git clone --recurse-submodules https://github.com/monatis/clip.cpp.git

In [ ]:
%cd clip.cpp
!mkdir build
%cd build

In [ ]:
!cmake -DCLIP_NATIVE=ON -DCLIP_BUILD_IMAGE_SEARCH=ON ..
!make

### Usage
- first download a quantized ggml model or build it your self using the information provided in the repository
-  now grab `the model`, `the image`, and the `text` you download to compare, Here is the Usage commands :

```bash
Usage: ./bin/main [options]                                                                                             
                                                                                                                        
Options:  -h, --help: Show this message and exit                                                                        
  -m <path>, --model <path>: path to model. Default: models/ggml-model-f16.bin                                          
  -t N, --threads N: Number of threads to use for inference. Default: 4                                                 
  --text <text>: Text to encode. At least one text should be specified                                                  
  --image <path>: Path to an image file. At least one image path should be specified                                    
  -v <level>, --verbose <level>: Control the level of verbosity. 0 = minimum, 2 = maximum. Default: 1                    
  ```

In [ ]:
# downloading pre-trained quantized GGML models from HF
!mkdir clip_models
!git clone https://huggingface.co/mys/ggml_CLIP-ViT-B-32-laion2B-s34B-b79K/ ./clip_models/

In [ ]:
!./bin/main \
 --model './clip_models/CLIP-ViT-B-32-laion2B-s34B-b79K_ggml-model-q5_1.gguf' \
 --image '/content/cat.jpg' \
 --text 'cat on a Turtle'

## Using the Python Bindings

In [ ]:
!pip install -U clip_cpp

### downloading pre-trained quantized GGML models from HF [**Old** Manual Download]

In [ ]:
## old Method to donwload models
# %cd /content/
# !git clone https://huggingface.co/Green-Sky/ggml_laion_clip-vit-b-32-laion2b-s34b-b79k/
# !mv ./ggml_laion_clip-vit-b-32-laion2b-s34b-b79k clip_models/

In [ ]:
from clip_cpp import Clip

In [ ]:
## handy cli command to show what are available models on HuggingFace that can be loaded
!clip-cpp-models

In [ ]:
# let's see the avaliable quantized models in the first repo-id
# as of v5.0 clip_cpp ✨✨ support for GGUF New Format
# something new here is the existance of solo text and image models along comined ones
!clip-cpp-models mys/ggml_CLIP-ViT-B-32-laion2B-s34B-b79K

In [ ]:
#downloading & loading the ggml model to Clip
model = Clip(
    model_path_or_repo_id="mys/ggml_CLIP-ViT-B-32-laion2B-s34B-b79K",
    model_file='CLIP-ViT-B-32-laion2B-s34B-b79K_ggml-model-q5_1.gguf',
    verbosity=2,
)

In [ ]:

text_2encode = 'cat on a Turtle'

tokens = model.tokenize(text_2encode)
text_embed = model.encode_text(tokens)

In [ ]:
## load and extract embedings of an image from the disk
image_2encode = '/content/cat.jpg'
image_embed = model.load_preprocess_encode_image(image_2encode)

In [ ]:
## perform similarity search between the image and the text
score = model.calculate_similarity(text_embed, image_embed)

# Alternatively, you can just do:
# score = model.compare_text_and_image(text, image_path)

print(f"Similarity score: {score}")

In [ ]:
text = 'dog eats a banana' ## unreleavant text compare to the image to see the score
score = model.compare_text_and_image(text, image_2encode);
print(f"Similarity score: {score}") # should be lower than the prev score

## More Real world Use-case example with fashion image dataset

In [ ]:
!pip install -q -U datasets

In [ ]:
from datasets import load_dataset

data = load_dataset(
    "ashraq/fashion-product-images-small",
    split="train"
)

In [ ]:
images = data["image"]

data = data.remove_columns("image")
product_frame = data.to_pandas()

In [ ]:
product_data = product_frame.reset_index(drop=True).to_dict(orient='index') # remove pandas default index and convert it to dict

In [ ]:
from pathlib import Path

#create an image dirctory to save the images to loaded with clip model
## ⚠️⚠️ currently clip Model it only Support loading images from disk not as PIL or numpy array for example
images_dir = Path('images')
images_dir.mkdir(exist_ok=True)

In [ ]:
image_files = []

##⚠️ it take about ~30 min to embed 5000 images of fashion dataset
## in the next cell you can download already 5000 extracted embeding inseated of waiting 😊
for idx, im in enumerate(images[:5000]):
    file_n = f'images/image_{idx}.jpeg'
    im.save(file_n, "JPEG")
    image_files.append(file_n)

In [ ]:
from tqdm import tqdm
import numpy as np

##⚠️it take about ~30 min to embed 5000 images of fashion dataset
# image_embeddings = [model.load_preprocess_encode_image(im) for im in tqdm(image_files)]
# image_embeddings = np.array(image_embeddings, dtype=np.float16)

# if you had already embedding saved as npy you can load it like this:
# image_embeddings = np.load('data.npy')

## 💡💡download & load already emded 5000 images from fashion dataset
## if you had already embedding saved as npy you can load it like this:
!wget "https://drive.google.com/uc?export=download&id=1kiewhrTHokuR7uuIYscOd3tGRVISXETF&confirm=yes" -O ./data.npy
image_embeddings = np.load('data.npy')

### saving the imbeddings to disk for faster retrieval

In [ ]:
np.save('data.npy', image_embeddings)

## USearch Vector Store for faster lookup

In [ ]:
!pip install -q -U usearch

In [ ]:
import numpy as np
from usearch.index import Index

In [ ]:
image_embeddings.shape # ndim is -> 512

In [ ]:
index = Index(
    ndim=512, # Define the number of dimensions in input vectors
    metric='cos', # Choose 'cos', 'l2sq', 'haversine' or other metric, default = 'ip'
    dtype='f16', # Quantize to 'f16' or 'i8' if needed, default = 'f32'
)

In [ ]:
## each vector should have unique corrsponding key hence range(...) mehtod
index.add(range(len(image_embeddings)), image_embeddings)

In [ ]:
def search_images_vecstore(query: str, index: Index) -> list:
    """
    extract query embeddings and run it against the usearch vector store
    return list of keys, distance (keys is the index of the crosponding image)
    """
    tokens = model.tokenize(query)
    query_embedded = np.array(model.encode_text(tokens))

    matches = index.search(query_embedded, count=10)
    return matches.to_list()


In [ ]:
from IPython.display import display


results = search_images_vecstore('blue bag', index)

for (img_idx, dist) in results:
    display(images[img_idx])

### Native lookup without vector store (it should be not as fast as vs)

In [ ]:
def semantic_search(query_embed: list[float], embeddings: list[list[float]], top_k: int = 5):
    scores = []
    for idx, image_embeddings in enumerate(embeddings):
        score = model.calculate_similarity(query_embed, image_embeddings)
        scores.append([score, idx])
    # Sort the list in descending order based on the score
    scores.sort(key = lambda x: x[0], reverse=True)
    return scores[:top_k]


def search_images(query: str, embeddings: list[list[float]], top_k: int=5):
    tokens = model.tokenize(query)
    query_embed = model.encode_text(tokens)

    results = semantic_search(query_embed, embeddings, top_k=top_k)
    return results

In [ ]:
x = search_images('blue bag', image_embeddings, top_k=10);x

In [ ]:
from IPython.display import display

for (score, im_idx) in x:
    display(images[im_idx])